# 🧪 GraphRAG test suite — 61 questions

Written against the seed corpus in Section B.3b of your notebook, so every question is
answerable and each one tests something specific.

**How to use:** copy the big code cell below into your notebook as a new cell at the very end,
then run the two cells after it. It defines nothing that connects on its own — the runners call
*your* `ask()`, `route()`, `neighbors()` and `find_node()`.

| Group | Count | What it proves |
|---|---|---|
| 🗄️ SQL | 8 | counts, sums, rankings go to the database, not to retrieval |
| 📄 RAG | 8 | single facts, including exact product codes the keyword leg must catch |
| 🕸️ GRAPH | 24 | multi-hop chains no single document contains |
| 🌍 GLOBAL | 8 | corpus-wide questions (needs Section H) |
| ⚠️ TRAPS | 7 | two similar figures, parent vs subsidiary, a score that does not exist |
| 🚫 REFUSALS | 6 | plausible, on-topic, and absent — the hardest kind to refuse |

Run in this order. The first two cost almost nothing, and if either fails, everything below it
fails for reasons that have nothing to do with retrieval.

```python
inspect_the_graph()      # free — is the graph built? can you reach the key chains?
test_routing()           # cheap — do questions reach the right engine?
test_graph_beats_rag()   # the money shot: same question, graph vs plain RAG
test_traps()             # does it stay careful when two figures look alike?
test_refusals()          # does it refuse, or invent?
test_global()            # needs build_communities() + summarize_communities()
run_all()                # everything, in order
```

---

### The four questions worth trying first

1. **"What is the relationship between Tidewater Frozen Holdings and SUP-1157?"**
   Should route to 🕸️ GRAPH and cite `SUBSIDIARY_OF` as `[graph]`. If not, fix the build first.

2. **"When does Tidewater's Stockton SQF certificate expire, and what happens if it lapses?"**
   The date is in the certificate register; the consequence is a contract clause that never says
   "SQF" or "Tidewater". Plain RAG finds the date and misses the consequence.

3. **"Acme Foods lost the produce RFP. Did they hold any food safety certifications?"**
   The sharpest test in the set. `SUP-1077` has **no** `HOLDS_CERTIFICATION` edge at all, while
   every other bidder has one — and both contracts require certification. No document says
   "Acme has no certificates." The **absence of an edge** is the finding.

4. **"Why does the Tidewater contract depend on a company that is not a party to it?"**
   Three hops: contract requires `CERT-HP-MSC` → held by `SUP-1157` → subsidiary of `SUP-1156`
   → the actual signatory. Nothing states this.

### The one diagnostic that matters most

If `neighbors("CERT-TW-SQF2-STK", hops=2)` does not show `CTR-2024-0489-TFH`, the flagship
multi-hop questions **cannot** work — and the problem is the graph BUILD (Sections G.3–G.5),
not retrieval, not the prompt, not the model.

In [ ]:
# ============================================================================
#  GraphRAG test suite — written against YOUR seed corpus (Section B.3b)
#
#  Paste as a new cell at the very END of the notebook, after Section H.
#  Then:
#      test_routing()        # cheap: checks WHERE each question goes. ~1 call each.
#      test_graph_beats_rag()# the money shot: same question, graph vs plain RAG
#      test_traps()          # the questions that catch hallucination
#      test_global()         # needs build_communities() + summarize_communities()
#      run_all()             # everything, in order
#
#  Every question below is answerable from the 16 seeded documents. The entities
#  are real: SUP-1042 Valle Verde Produce, SUP-1077 Acme Foods, SUP-1103 Globex
#  Dairy, SUP-1156 Tidewater Frozen Holdings, SUP-1189 Northlake Protein,
#  SUP-1157 (Tidewater's subsidiary), and 137 relationship triples between them.
# ============================================================================

# ---------------------------------------------------------------------------
#  A. THE CHAINS YOUR GRAPH ACTUALLY CONTAINS
#     Read these first — they tell you WHY each question below is a fair test.
#
#   1. CERTIFICATE → CONTRACT
#        CERT-TW-SQF2-STK  --COVERS_FACILITY-->  FAC-TW-STOCKTON
#        CERT-TW-SQF2-STK  --REQUIRED_BY----->   CTR-2024-0489-TFH
#        SUP-1156          --OPERATES_FACILITY-> FAC-TW-STOCKTON
#        SUP-1156          --PARTY_TO--------->  CTR-2024-0489-TFH
#      A certificate's expiry date and the contract clause that depends on it
#      live in DIFFERENT documents that share almost no vocabulary.
#
#   2. THE SUBSIDIARY CHAIN  ← the best test in the whole corpus
#        SUP-1157  --SUBSIDIARY_OF-->  SUP-1156      (Tidewater owns it)
#        SUP-1156  --ACQUIRED------->  SUP-1157
#        SUP-1156  --LIABLE_FOR----->  SUP-1157
#        SUP-1157  --HOLDS_CERTIFICATION--> CERT-HP-MSC
#        CERT-HP-MSC --REQUIRED_BY--> CTR-2024-0489-TFH
#      A contract signed by the PARENT depends on a certificate held by the
#      SUBSIDIARY. Three hops. No single document states this.
#
#   3. THE COMPETITIVE CHAIN
#        BID-2024-089 (Valle Verde) SCORED 91.4, AWARDED, → CTR-2024-0412-VV
#        BID-2024-091 (Acme)        SCORED 74.7, LOST_TO BID-2024-089
#
#   4. THE PERFORMANCE CHAIN
#        ORG-RUSD --REQUESTED_CORRECTIVE_ACTION_FROM--> SUP-1156
#        SUP-1156 --SATISFACTION_RATING--> 4.1/5   (the lowest of five)
#        SUP-1156 --DECLINED_TO_BID--> RFP-2024-0412 and RFP-2024-0455
# ---------------------------------------------------------------------------


TESTS = [
    # ======================= SQL — computed from columns =======================
    dict(q="How many RFPs are there in total?",
         expect="sql",
         why="A count over rows. Nothing in a document states this number.",
         look_for="a number, returned as a DataFrame"),

    dict(q="Which supplier has the highest total bid amount, and in which category?",
         expect="sql",
         why="Ranking + aggregation. The classic SQL shape.",
         look_for="a table, not prose"),

    dict(q="List all bids submitted after March 2024, newest first.",
         expect="sql",
         why="A filtered, ordered list of records.",
         look_for="rows with dates in order"),

    # ======================= RAG — one fact, one document =====================
    dict(q="What are Globex Dairy Cooperative's payment terms?",
         expect="rag",
         why="A single fact stated in prose in one document.",
         look_for="the terms, with a [n] citation"),

    dict(q="What does bid BID-2024-089 quote for product VV-PRD-1180?",
         expect="rag",
         why="Exact-token lookup — this is where the KEYWORD leg earns its place. "
             "Pure vector search struggles with codes like VV-PRD-1180.",
         look_for="a price or spec, cited"),

    dict(q="What is the delivery window specified in RFP-2024-0412?",
         expect="rag",
         why="One clause, one document.",
         look_for="a window, cited"),

    # ======================= GRAPH — the multi-hop questions ==================
    dict(q="When does Tidewater's Stockton SQF certificate expire, and what happens "
           "if it lapses?",
         expect="graph",
         why="CHAIN 1. The expiry date is in the certificate register; the consequence "
             "is a clause in CTR-2024-0489-TFH that never mentions 'SQF' or 'Tidewater'. "
             "Plain RAG finds the date and misses the consequence.",
         look_for="BOTH the expiry date AND the contractual consequence"),

    dict(q="Why does the Tidewater contract depend on a company that is not a party to it?",
         expect="graph",
         why="CHAIN 2 — the hardest question in the corpus. Requires: contract requires "
             "CERT-HP-MSC → held by SUP-1157 → subsidiary of SUP-1156 → which is the "
             "contracting party. Three hops, no single document.",
         look_for="the subsidiary relationship AND the certificate dependency"),

    dict(q="What is the relationship between Tidewater Frozen Holdings and SUP-1157?",
         expect="graph",
         why="Names two entities and asks how they relate — the textbook graph shape.",
         look_for="SUBSIDIARY_OF / ACQUIRED / LIABLE_FOR, cited as [graph]"),

    dict(q="What happens to the Valleverde contract if their cold-chain certification lapses?",
         expect="graph",
         why="Consequence question: CERT-VV-SQF2 --REQUIRED_BY--> CTR-2024-0412-VV.",
         look_for="the contractual consequence, not just the certificate"),

    dict(q="Which facilities would be affected if Tidewater lost its Stockton certification, "
           "and which contracts depend on those facilities?",
         expect="graph",
         why="Two hops outward, then back into contracts. Tests that expansion is "
             "reaching hop 2 and not stopping at hop 1.",
         look_for="FAC-TW-STOCKTON and CTR-2024-0489-TFH both named"),

    dict(q="Everything we know about Tidewater Frozen Holdings.",
         expect="graph",
         why="'Everything about X' where X appears across bids, contracts, certificates, "
             "facilities and a corrective-action notice. Tests breadth of expansion.",
         look_for="facts from at least three different document types"),

    dict(q="Which supplier lost the produce RFP, and what did the winner score?",
         expect="graph",
         why="CHAIN 3. LOST_TO connects two bids that are described in separate documents.",
         look_for="Acme (74.7) lost to Valle Verde (91.4)"),

    dict(q="Why did Riverside request corrective action, and did that supplier bid on "
           "anything afterwards?",
         expect="graph",
         why="CHAIN 4. Corrective action, the rating, and two DECLINED_TO_BID edges — "
             "three documents, one causal story.",
         look_for="the corrective action AND the declined bids"),

    dict(q="Can Central Valley USD buy under Riverside's produce contract, and what "
           "conditions apply?",
         expect="graph",
         why="MAY_PIGGYBACK_ON — a relationship stated in one document with conditions "
             "stated in another.",
         look_for="the piggyback right AND its conditions"),

    # ======================= GLOBAL — needs Section H =========================
    dict(q="What are the recurring compliance and delivery risks across all suppliers?",
         expect="global",
         why="No chunk contains this. It only exists as a pattern across the corpus.",
         look_for="themes spanning several suppliers, not one supplier's detail"),

    dict(q="What are the main themes in how suppliers respond to our RFPs?",
         expect="global",
         why="'Main themes' + 'suppliers' plural — corpus-wide by construction.",
         look_for="patterns, each attributed to named suppliers"),

    dict(q="Overall, what is this procurement programme most exposed to?",
         expect="global",
         why="A question about the whole corpus with no named entity at all.",
         look_for="risks synthesised across communities"),

    dict(q="Summarise the certification landscape across all our suppliers.",
         expect="global",
         why="Requires seeing every certificate at once, not retrieving one.",
         look_for="SQF, GAP, BRCGS, MSC and who holds what"),

    # ======================= TRAPS — where systems hallucinate ================
    dict(q="What is our supplier satisfaction rating?",
         expect=None,
         why="⚠️ THE TRAP. 4.8/5 is ONE supplier's rating (Valle Verde). There is also a "
             "district-wide index of 8.5/10. A careless system merges them or reports "
             "one as the other.",
         look_for="BOTH figures, each attributed — or a request to clarify. "
                  "A single blended number is a FAILURE."),

    dict(q="What certifications does Tidewater hold?",
         expect=None,
         why="⚠️ TRAP. Tidewater (SUP-1156) holds certs for its own facilities; CERT-HP-MSC "
             "belongs to its SUBSIDIARY SUP-1157. Attributing the subsidiary's certificate "
             "to the parent is wrong, and easy to do.",
         look_for="Tidewater's own certificates, with the subsidiary's noted SEPARATELY"),

    dict(q="What was Acme's score on the produce RFP?",
         expect=None,
         why="⚠️ TRAP. 74.7 belongs to BID-2024-091. 91.4 belongs to Valle Verde. Systems "
             "that retrieve both documents often return the wrong number.",
         look_for="74.7, attributed to Acme — not 91.4"),

    dict(q="Which supplier operates the Oxnard facility?",
         expect=None,
         why="⚠️ TRAP. FAC-HP-OXNARD is operated by SUP-1157, the subsidiary — not by "
             "Tidewater, even though Tidewater owns SUP-1157.",
         look_for="SUP-1157, with the ownership noted"),

    # ======================= REFUSALS — should NOT invent =====================
    dict(q="What is the capital of France?",
         expect="none",
         why="General knowledge. Must be refused, not answered.",
         look_for="a refusal and a hint about what CAN be asked"),

    dict(q="What were our supplier ratings in 2019?",
         expect=None,
         why="Plausible, on-topic, and absent from the corpus. The hardest refusal.",
         look_for='exactly the REFUSAL string: "I don\'t have that in the indexed documents."'),

    dict(q="Which supplier has the best cybersecurity posture?",
         expect=None,
         why="On-topic vocabulary, no such data anywhere. Tests the relevance floor.",
         look_for="a refusal, NOT an inference from unrelated documents"),

    dict(q="Hello, how are you?",
         expect="none",
         why="Chit-chat. Should be refused cleanly by the router.",
         look_for="the 'ask something about your tables or documents' message"),
]


# ---------------------------------------------------------------------------
#  RUNNERS
# ---------------------------------------------------------------------------
def _actual_engine(q):
    """Where would ask() send this? Mirrors ask()'s own logic, without answering."""
    try:
        if _is_global(q, verbose=False):          # Section H, if installed
            return "global"
    except NameError:
        pass
    try:
        return route(q)["engine"]
    except Exception as e:
        return f"ERROR:{type(e).__name__}"


def test_routing(only=None):
    """Cheap. Checks WHERE each question goes, without paying to answer it.

    Run this FIRST. If routing is wrong, every answer below it is wrong for a
    reason that has nothing to do with retrieval.
    """
    rows, wrong = [], 0
    tests = [t for t in TESTS if t["expect"]] if only is None else \
            [t for t in TESTS if t["expect"] == only]
    print(f"Routing {len(tests)} question(s)…\n")
    for t in tests:
        got = _actual_engine(t["q"])
        ok = (got == t["expect"])
        wrong += (not ok)
        rows.append({"expected": t["expect"], "got": got, "ok": "✅" if ok else "❌",
                     "question": t["q"][:66]})
    df = pd.DataFrame(rows)
    display(df)
    print(f"\n{len(tests) - wrong}/{len(tests)} routed as expected.")
    if wrong:
        print("A wrong route is usually one of three things:")
        print("  • the router crashed and failed open to RAG — look for ⚠️ ROUTER FAILED")
        print("  • the graph is not built, so route() rewrites 'graph' to 'rag'")
        print("  • no community summaries exist, so global questions fall back")
    return df


def test_graph_beats_rag():
    """The comparison that proves the graph is doing something.

    Same question, both engines. If the answers are identical, the graph is not
    earning its keep FOR THAT QUESTION — which is real information, not a bug.
    """
    pairs = [t for t in TESTS if t["expect"] == "graph"][:4]
    for t in pairs:
        print("\n" + "=" * 78)
        print("Q:", t["q"])
        print("   look for:", t["look_for"])
        print("=" * 78)
        print("\n--- 🕸️  GRAPH ---")
        g = ask(t["q"], mode="graph", verbose=False)
        print("\n--- 📄 PLAIN RAG ---")
        show(ask_rag(t["q"]))
        print("\n💡 Did the graph answer contain something RAG's did not?")
        print("   Look for sources tagged '← pulled in by the graph'. If none ever")
        print("   appear, expansion is contributing nothing.")


def test_traps():
    """The questions that separate a careful system from a confident one."""
    for t in [x for x in TESTS if "TRAP" in x["why"]]:
        print("\n" + "=" * 78)
        print("Q:", t["q"])
        print("⚠️ ", " ".join(t["why"].split())[:200])
        print("   PASS =", t["look_for"])
        print("=" * 78)
        ask(t["q"])


def test_refusals():
    """A system that never refuses is not safe — it is just confident."""
    for t in [x for x in TESTS if "refus" in x["look_for"].lower()
              or "Refus" in x["why"] or "absent" in x["why"]]:
        print("\n" + "-" * 78)
        print("Q:", t["q"], "\n   PASS =", t["look_for"])
        ask(t["q"])


def test_global():
    """Needs build_communities() + summarize_communities() to have been run."""
    try:
        with db(dict_rows=True) as cur:
            cur.execute(f"SELECT count(*) AS n FROM {COMMUNITY_TABLE} "
                        f"WHERE summary IS NOT NULL")
            n = cur.fetchone()["n"]
    except Exception:
        print("Section H is not installed — no global search in this notebook.")
        return
    if not n:
        print("No community summaries yet. Run build_communities() then "
              "summarize_communities() first.")
        return
    for t in [x for x in TESTS if x["expect"] == "global"]:
        print("\n" + "=" * 78)
        print("Q:", t["q"], "\n   look for:", t["look_for"])
        print("=" * 78)
        ask(t["q"], mode="global")


def inspect_the_graph():
    """Before blaming retrieval, look at what the graph actually knows."""
    print("── Is the graph even built? ──")
    with db(dict_rows=True) as cur:
        for label, t in [("nodes", NODE_TABLE), ("edges", EDGE_TABLE),
                         ("mentions", MENTION_TABLE)]:
            cur.execute(f"SELECT count(*) AS n FROM {t}")
            print(f"   {label:<10} {cur.fetchone()['n']:>7,}")
        cur.execute(f"""SELECT predicate, count(*) AS n FROM {EDGE_TABLE}
                        GROUP BY predicate ORDER BY n DESC LIMIT 12""")
        print("   top predicates:",
              ", ".join(f"{r['predicate']}={r['n']}" for r in cur.fetchall()))
        cur.execute(f"""SELECT method, count(*) AS n FROM {MENTION_TABLE}
                        GROUP BY method ORDER BY n DESC""")
        print("   mentions by method:",
              ", ".join(f"{r['method']}={r['n']}" for r in cur.fetchall()))
        print("   ↑ mostly 'label' means entity linking is guesswork — expect noise.")

    print("\n── Can you reach the key chains? ──")
    print("find_node('Tidewater'):");            find_node("Tidewater")
    print("\nneighbors('SUP-1156', hops=2):");   neighbors("SUP-1156", hops=2)
    print("\nneighbors('CERT-TW-SQF2-STK', hops=2):")
    neighbors("CERT-TW-SQF2-STK", hops=2)
    print("\n↑ If CTR-2024-0489-TFH does NOT appear within 2 hops of that certificate,")
    print("  the flagship question CANNOT work, and the problem is the BUILD")
    print("  (Sections G.3–G.5), not retrieval or the model.")


def run_all():
    print("=" * 78); print("1 · IS THE GRAPH BUILT?"); print("=" * 78)
    inspect_the_graph()
    print("\n" + "=" * 78); print("2 · ROUTING (cheap)"); print("=" * 78)
    test_routing()
    print("\n" + "=" * 78); print("3 · GRAPH vs PLAIN RAG"); print("=" * 78)
    test_graph_beats_rag()
    print("\n" + "=" * 78); print("4 · TRAPS"); print("=" * 78)
    test_traps()
    print("\n" + "=" * 78); print("5 · REFUSALS"); print("=" * 78)
    test_refusals()
    print("\n" + "=" * 78); print("6 · GLOBAL"); print("=" * 78)
    test_global()


print(f"✅ {len(TESTS)} test questions loaded.")
print("   inspect_the_graph()    ← start here, costs nothing")
print("   test_routing()         ← then this")
print("   test_graph_beats_rag() · test_traps() · test_refusals() · test_global()")
print("   run_all()")


# ============================================================================
#  PART 2 — 33 further questions
#
#  These exercise corners the first set does not: lots, people, the email
#  thread, products, the piggyback clause, and the two most interesting facts
#  hiding in your corpus —
#
#    • SUP-1077 (Acme Foods) holds NO certifications at all. It also lost.
#      Nothing states those two facts together; the graph is the only way there.
#    • SUP-1157 performs part of CTR-2024-0489-TFH but is not a party to it.
#      Its parent SUP-1156 signed. The subsidiary does the work.
# ============================================================================

TESTS += [
    # ======================= SQL =======================
    dict(q="How many bids did we receive for each RFP?",
         expect="sql",
         why="GROUP BY over records. No document states a per-RFP count.",
         look_for="one row per RFP with a count"),

    dict(q="What is the total value of all awarded contracts?",
         expect="sql",
         why="A SUM. The classic case where retrieval can only find text near a number.",
         look_for="a single total, as a table"),

    dict(q="Show every bid with its amount, highest first.",
         expect="sql",
         why="Ordered list of records, straight from columns.",
         look_for="a sorted DataFrame"),

    dict(q="How many suppliers do we have on file?",
         expect="sql",
         why="A count. If this routes to RAG, the router is broken.",
         look_for="one number"),

    dict(q="Which category has the most RFPs?",
         expect="sql",
         why="Group, count, rank — three SQL verbs in one question.",
         look_for="a category and a count"),

    # ======================= RAG =======================
    dict(q="What insurance requirements does RFP-2024-0489 specify?",
         expect="rag",
         why="One clause, one document, stated in prose.",
         look_for="the requirement with a [n] citation"),

    dict(q="What did Addendum No. 1 change about product substitutions?",
         expect="rag",
         why="A specific change in a specific document.",
         look_for="the substitution rule, cited"),

    dict(q="What price did BID-2024-104 quote for GX-DRY-0101?",
         expect="rag",
         why="Exact product code — the keyword leg should win this, not vectors.",
         look_for="a price tied to that exact SKU"),

    dict(q="What are the delivery penalties in contract CTR-2024-0489-TFH?",
         expect="rag",
         why="A contract clause. Single document, single fact.",
         look_for="the penalty terms, cited"),

    dict(q="What is the bid submission deadline in RFP-2024-0455?",
         expect="rag",
         why="A date stated once in one document.",
         look_for="a date, cited"),

    # ======================= GRAPH =======================
    dict(q="Acme Foods lost the produce RFP. Did they hold any food safety certifications?",
         expect="graph",
         why="★ THE BEST TEST IN PART 2. SUP-1077 has NO HOLDS_CERTIFICATION edge at all, "
             "while every other bidder has one. Both contracts REQUIRE certifications. "
             "No document says 'Acme has no certificates' — the ABSENCE of an edge is the "
             "finding, and only a graph view surfaces it.",
         look_for="that Acme holds none, contrasted with the certified competitors"),

    dict(q="Which supplier is performing part of a contract it never signed?",
         expect="graph",
         why="SUP-1157 --FULFILS_CATEGORY_UNDER--> CTR-2024-0489-TFH, but SUP-1156 is the "
             "PARTY_TO. Parent signs, subsidiary delivers.",
         look_for="SUP-1157 doing the work, SUP-1156 as the signatory"),

    dict(q="Which lots of RFP-2024-0489 did both Tidewater and Northlake bid on?",
         expect="graph",
         why="BID-117 bids lots 1+2, BID-118 bids lots 2+3. The overlap is LOT-0489-2, and "
             "it is computed from edges in two different bid documents.",
         look_for="Lot 2 as the contested one"),

    dict(q="Tidewater operates two facilities. Does the Riverside contract depend on both?",
         expect="graph",
         why="CERT-TW-SQF2-STK (Stockton) is REQUIRED_BY the contract; CERT-TW-SQF2-BKF "
             "(Bakersfield) is not. A subtle distinction that matters commercially.",
         look_for="only Stockton being contractually required"),

    dict(q="Who recommended the Valle Verde award, and what else has that person done?",
         expect="graph",
         why="PER-DWHITFIELD RECOMMENDED the bid, AUTHORED the performance review, APPROVES "
             "substitutions, and SENT_MESSAGE_IN the corrective-action thread. Four roles "
             "across four documents.",
         look_for="Whitfield, with at least three of the four roles"),

    dict(q="What was the December 2024 email thread about, and which contract does it affect?",
         expect="graph",
         why="THR-2024-0912 CONCERNS both CTR-2024-0489-TFH and CERT-TW-SQF2-STK — an email "
             "linked to a contract and a certificate.",
         look_for="the corrective action, the contract, and the certificate"),

    dict(q="Which suppliers were invited to bid but never submitted one?",
         expect="graph",
         why="SOLICITS edges minus SUBMITTED edges. SUP-1156 has DECLINED_TO_BID twice; "
             "SUP-1204 was solicited and simply never appears as a bidder. Set arithmetic "
             "over edges — impossible from any single passage.",
         look_for="Tidewater's declines, and any solicited supplier with no bid"),

    dict(q="Has Acme Foods ever been formally evaluated by Riverside?",
         expect="graph",
         why="EVALUATED covers SUP-1042, 1103, 1156 and 1189 — not SUP-1077. Another "
             "absent-edge finding.",
         look_for="that Acme is missing from the evaluated set"),

    dict(q="Trace the path from RFP-2024-0412 to the contract that resulted from it.",
         expect="graph",
         why="RFP → BID-2024-089 → CTR-2024-0412-VV, three documents, two RESULTED_IN / "
             "ARISES_FROM hops. Explicitly asks for a chain.",
         look_for="all three ids in the right order"),

    dict(q="If Central Valley USD purchases under the Valle Verde contract, which "
           "certifications must stay valid?",
         expect="graph",
         why="MAY_PURCHASE_UNDER → CTR-2024-0412-VV → REQUIRES_CERTIFICATION → CERT-VV-GAP "
             "and CERT-VV-SQF2. Three hops from an organisation to two certificates.",
         look_for="both CERT-VV-GAP and CERT-VV-SQF2"),

    dict(q="Why did Tidewater decline two RFPs but bid on the third?",
         expect="graph",
         why="DECLINED_TO_BID on 0412 (produce) and 0455 (dairy), SUBMITTED on 0489 (frozen). "
             "The reason is category fit, stated in the qualification profile — a different "
             "document from the RFPs.",
         look_for="the category logic, drawn from more than one document"),

    dict(q="Which products did the winning produce bid quote?",
         expect="graph",
         why="AWARDED → BID-2024-089 → QUOTES_PRODUCT → VV-PRD-1180, VV-PRD-1204. You must "
             "know which bid won before you know which products to look up.",
         look_for="both product codes, and that this was the winning bid"),

    dict(q="What connects the corrective action notice to the frozen foods contract?",
         expect="graph",
         why="Corrective action → SUP-1156 → PARTY_TO → CTR-2024-0489-TFH, with the email "
             "thread as corroboration. A 'what connects X and Y' question.",
         look_for="a named chain, not a restatement of one document"),

    dict(q="Which contracts would be at risk if the Oxnard facility lost its certification?",
         expect="graph",
         why="FAC-HP-OXNARD → CERTIFIED_BY → CERT-HP-MSC → REQUIRED_BY → CTR-2024-0489-TFH. "
             "The facility belongs to the subsidiary; the contract belongs to the parent.",
         look_for="CTR-2024-0489-TFH, via the subsidiary"),

    dict(q="Who are the supplier-side contacts, and which companies do they work for?",
         expect="graph",
         why="WORKS_FOR spans seven people across five organisations, described in separate "
             "documents.",
         look_for="several people each correctly matched to their employer"),

    # ======================= GLOBAL =======================
    dict(q="What patterns do you see in why bids lose?",
         expect="global",
         why="Needs every bid outcome at once. No chunk compares losses.",
         look_for="patterns spanning more than one losing bid"),

    dict(q="Across all our contracts, which certification types matter most?",
         expect="global",
         why="'Across all' + a distribution question. Corpus-wide by construction.",
         look_for="SQF prominent, with GAP / BRCGS / MSC placed around it"),

    dict(q="What are the common characteristics of suppliers we rate highly?",
         expect="global",
         why="Requires comparing five suppliers' ratings, certificates and outcomes together.",
         look_for="traits shared by the 4.5+ suppliers"),

    dict(q="Overall, how concentrated is our supplier risk?",
         expect="global",
         why="A judgement about the whole portfolio. Nothing states it.",
         look_for="concentration reasoning across suppliers and categories"),

    # ======================= TRAPS =======================
    dict(q="Which certificate does the Riverside-Tidewater contract require?",
         expect=None,
         why="⚠️ TRAP. It requires TWO: CERT-TW-SQF2-STK and CERT-HP-MSC. Naming only one "
             "is a plausible, confident, incomplete answer — the most dangerous kind.",
         look_for="BOTH certificates. One is a FAILURE, however well written."),

    dict(q="How many facilities does Tidewater operate?",
         expect=None,
         why="⚠️ TRAP. SUP-1156 operates two (Stockton, Bakersfield). Its subsidiary SUP-1157 "
             "operates a third (Oxnard). Answering '3' silently merges parent and subsidiary.",
         look_for="two, with the subsidiary's third noted separately"),

    dict(q="What score did the winning frozen foods bid receive?",
         expect=None,
         why="⚠️ TRAP. SCORED exists only for the produce bids (91.4 and 74.7). BID-2024-117 "
             "has no score anywhere. The tempting failure is to borrow 91.4.",
         look_for="an admission that no score is recorded — NOT a number"),

    # ======================= REFUSALS =======================
    dict(q="What is Valle Verde's employee headcount?",
         expect=None,
         why="On-topic, about a real supplier, and simply not in the corpus.",
         look_for="the refusal string, not an estimate"),

    dict(q="Which supplier has the lowest carbon footprint?",
         expect=None,
         why="Sustainability vocabulary overlaps with food-safety language, so retrieval "
             "will surface certificate documents. It must still refuse.",
         look_for="a refusal, NOT an inference from SQF or GAP certificates"),
]

print(f"✅ Part 2 loaded — {len(TESTS)} test questions in total.")


In [ ]:
inspect_the_graph()   # free — start here

In [ ]:
test_routing()        # cheap — then this

---

## Grading

`test_routing()` is objective — expected engine vs actual, printed as a table.

The rest need your eye. Each question carries a `look_for` field saying what a correct answer
contains. Two failure modes matter more than any other:

**Confident incompleteness.** *"Which certificate does the Riverside–Tidewater contract
require?"* has **two** right answers (`CERT-TW-SQF2-STK` and `CERT-HP-MSC`). A fluent answer
naming one is a failure, and it will not look like one.

**Borrowed figures.** *"What score did the winning frozen foods bid receive?"* has **no**
answer — only the produce bids were scored (91.4 and 74.7). If a number comes back, the system
borrowed it from a neighbouring document.

And whenever a graph answer disappoints, check the sources for `← pulled in by the graph`
before blaming the model. If that tag never appears, expansion contributed nothing and the
problem is upstream.